In [ ]:
# Install BERTopic
%pip install bertopic

# Choose an embedding backend
%pip install bertopic[flair, gensim, spacy, use]

# Topic modeling with images
%pip install bertopic[vision]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'bertopic[flair,'

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Extract topics from the well-known 20 newsgroups dataset

In [2]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
docs = docs[:2000]

Clean the text

In [3]:
import re # regular expression module : search, replace and remove patterns in text

def clean(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", " ", text)          
    # remove URLs
    text = re.sub(r"\b\w+@\w+\.\w+\b", " ", text)          
    # remove emails
    text = re.sub(r"\[deleted\]|\[removed\]", " ", text, flags=re.I)
    text = re.sub(r"[\*\_`>#]", " ", text)                 
    # light markdown chars
    text = re.sub(r"\s+", " ", text)                       
    # collapse whitespace
    return text.strip()
    # removes leading or trailing spaces

cleaned_docs = []

for d in docs:
    if isinstance(d, str) and len(d) > 50:
        cleaned_docs.append(clean(d))

docs = cleaned_docs

Fit BASE model
- note that the embedding (converting text into vectors), dimensionality reduction, and clustering steps were not explicitly written 
- fit_transform() automatically ran UMAP (dimensionality reduction) and HDBSCAN (clustering) c-TF-IDF words (representation method) by default

In [4]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

base_model = BERTopic(
    min_topic_size=10,
    umap_model=UMAP(n_neighbors=10, min_dist=0.0),
    hdbscan_model=HDBSCAN(min_cluster_size=10)
)

base_topics, _ = base_model.fit_transform(docs)

c:\Users\xinle\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 579.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Access the frequent topics that were generated
- Top keywords that define the topic [the, to, of, ...] show that it is dominated by common words (i.e. not meaningful) -> stopwords weren't removed strongly for topic representation / need to tune the vectoriser / representation

In [5]:
base_model.get_topic_info().head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,47,-1_the_to_and_in,"[the, to, and, in, of, team, that, have, is, for]",[Hindsight's 20-20. Nobody expected Kisio to h...
1,0,1731,0_the_to_of_and,"[the, to, of, and, is, in, that, it, for, you]",[% %By Elias Davidsson - April 1991 (Revision ...
2,1,65,1_the_he_to_and,"[the, he, to, and, in, that, his, year, of, is]",[I'm not quite sure how these numbers are gene...
3,2,27,2_to_the_games_and,"[to, the, games, and, game, on, in, espn, is, ...","[So this year you had a game on SCNY, a game o..."
4,3,25,3_the_to_was_and,"[the, to, was, and, he, that, of, is, in, game]","[I think that you are incorrect, Roger. Patric..."
5,4,22,4_10_shots_period_pp,"[10, shots, period, pp, saves, 11, the, det, p...",[NHL RESULTS FOR GAMES PLAYED 4/05/93. -------...


Add proper vectoriser (stopwords + bigrams + min_df)

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

Refit with better representation model
- Vectoriser removes English stopwords + allows bigrams -> effects keyword extraction quality
- Representation model (KeyBERTInspired) re-ranks topic words using embedding similarity
- Smaller clusters are discouraged / merged because of min topic size

In [9]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

improved_model = BERTopic(
    vectorizer_model=vectorizer_model,
    representation_model=KeyBERTInspired(),
    min_topic_size=10
)

topics, _ = improved_model.fit_transform(docs)
improved_model.get_topic_info()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 431.77it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,Representative_Docs
0,0,1730,0_bus_hardware_drivers_driver,"[bus, hardware, drivers, driver, armenians, ca...","[Folks, It's time to start building some preci..."
1,1,187,1_replay_espn_montreal_flyers,"[replay, espn, montreal, flyers, red wings, pu...",[NHL PLAYOFF RESULTS FOR 4/19/93. ------------...
